In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

silver_folder = 'abfss://silver@asterisktotle01.dfs.core.windows.net/PHHousing'

gold_folder = 'abfss://gold@asterisktotle01.dfs.core.windows.net/PHHousing'

housing_silver = spark.read.format('delta').load(silver_folder + '/housing_silver')

dim_foreclosure = (
    housing_silver.select("sourceSlug", "sourceName").distinct()
    .withColumn("channel",
        F.when(F.lower("sourceSlug").contains("pagibig"), "Pag-IBIG")
         .otherwise("Bank (general)"))
    .withColumn("lenderSk", F.row_number().over(Window.orderBy("sourceSlug")))
    .select("lenderSk", "sourceSlug", "sourceName", "channel")
)

# housing_fk.write.format("delta").mode("overwrite").save(silver_folder + '/housing_silver')
dim_foreclosure.write.format("parquet").mode("overwrite").save(gold_folder + "/dim_foreclosure")
print('dim_foreclosure written to gold layer')



In [0]:
dim_fam = spark.read.format('delta').load(silver_folder + '/dim_fam')
# 2. Process and Transform Pipeline
fact_household = (
    dim_fam.drop('avgIncome', 'avgExpenses')
      .withColumn('capacityToPayPagibig', (F.col('monthlyIncome') * 0.35).cast('decimal(18,2)'))
      .withColumn('capacityToPayBank', (F.col('monthlyIncome') * 0.30).cast('decimal(18,2)'))
      .withColumn(
          'reliability',
          F.when(F.col('families') >= 200, 'High')
           .when(F.col('families') >= 100, 'Moderate')
           .when(F.col('families') >= 30, 'Low')
           .otherwise('Very Low')
      )
)

fact_household = fact_household.select(
    'geographyFk',
    'families',
    'reliability',
    'monthlyIncome',
    'monthlyExpenses',
    'netIncome',
    'capacityToPayPagibig',
    'capacityToPayBank'
    )

fact_household = fact_household.filter(F.col('geographyFk').isNotNull())
fact_household.write.format('parquet').mode('overwrite').save(gold_folder + '/fact_household')

# fact_household.display()



In [0]:
silver_dim_geography = spark.read.format('delta').load(silver_folder + '/dim_geography')

gold_dim_geography = silver_dim_geography \
    .withColumnRenamed('islandGroupCode', 'islandGroup') \
    .withColumnRenamed('geographyFk', 'geographyId') \
    .select(
    'geographyId',
    'geographyLevel',
    'geographyName',
    'provinceCode',
    'provinceName',
    'islandGroup'
)

gold_dim_geography.write.format('parquet').mode('overwrite').save(gold_folder + '/dim_geography')


display(gold_dim_geography)



In [0]:
dim_lender_terms = spark.createDataFrame([
    ("Pag-IBIG",       0.90, 0.0575, 0.35, 10_000_000, 30, "official"),
    ("Bank (general)", 0.80, 0.0700, 0.30, None,        20, "estimated"),
], ["channel", "ltv_pct", "interest_rate", "dti_cap_pct",
    "max_loan_amount", "max_term_years", "source_confidence"])

display(dim_lender_terms)

In [0]:
dim_geography = spark.read.format('delta').load(silver_folder + '/dim_geography')
median_price_df = (
    housing_silver
    .join(dim_geography.select("geographyFk"), "geographyFk", "left")
    .groupBy("geographyFk")
    .agg(
        F.expr("percentile_approx(price, 0.5)").alias("houseMedianPrice"),
        F.count("*").alias("listingCount")
    )
)

display(median_price_df)



In [0]:
terms = {row["channel"]: row.asDict() for row in dim_lender_terms.collect()}
pagibig = terms["Pag-IBIG"]
bank = terms["Bank (general)"]



def monthly_pmt(principal_col, annual_rate, years):
    r = annual_rate / 12
    n = years * 12
    pmt = principal_col * r * F.pow(F.lit(1 + r), n) / (F.pow(F.lit(1 + r), n) - 1)
    return pmt.cast("decimal(18,2)")
fact_affordability = (
    median_price_df
    .join(fact_household, "geographyFk", "inner")
    .withColumn("pagIbigRequiredLoan", F.col("houseMedianPrice") * pagibig["ltv_pct"])
    .withColumn("pagIbigAmmortization",
                monthly_pmt(F.col("pagIbigRequiredLoan"), pagibig["interest_rate"], pagibig["max_term_years"]))
    .withColumn("pagIbigCapacity", (F.col("monthlyIncome") * pagibig["dti_cap_pct"]).cast("decimal(18,2)"))
    .withColumn("pagIbigGap", F.round(F.col("pagIbigAmmortization") - F.col("pagIbigCapacity"), 2))
    .withColumn("pagIbigLoanStatus", F.when(F.col("pagIbigGap") <= 0, "Applicable").otherwise("Not Applicable"))

    .withColumn("bankRequiredLoan", F.col("houseMedianPrice") * bank["ltv_pct"])
    .withColumn("bankAmmortization",
                monthly_pmt(F.col("bankRequiredLoan"), bank["interest_rate"], bank["max_term_years"]))
    .withColumn("bankCapacity", (F.col("monthlyIncome") * bank["dti_cap_pct"]).cast("decimal(18,2)"))
    .withColumn("bankGap", F.round(F.col("bankAmmortization") - F.col("bankCapacity"), 2))
    .withColumn("bankLoanStatus", F.when(F.col("bankGap") <= 0, "Applicable").otherwise("Not Applicable"))
)

fact_affordability = fact_affordability.select(
    'geographyFk',
    'houseMedianPrice',
    'listingCount',
    'pagIbigRequiredLoan',
    'pagIbigAmmortization',
    'pagIbigCapacity',
    'pagIbigGap',
    'pagIbigLoanStatus',
    'bankRequiredLoan',
    'bankAmmortization',
    'bankCapacity',
    'bankGap',
    'bankLoanStatus'

)

fact_affordability.write.format("parquet").save(gold_folder + "/fact_affordability")

In [0]:
fact_housing = housing_silver.select(
    "id",
    "geographyFk",
    "title",
    "price",
    "pricePerSqm",
    "floorArea",
    "lotArea",
    "daysListed",
    "firstSeenAt"
)

fact_housing = fact_housing.filter(F.col("geographyFk").isNotNull())

fact_housing.write.mode("overwrite").format("parquet").save(gold_folder + "/fact_housing")
print("fact_housing table written to gold layer")

In [0]:
fact_housing.display()